In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1) Load Excel file
# -------------------------------
file_path = "complex_multi_product_order_forecasting_dataset.xlsx"

daily_df = pd.read_excel(file_path, sheet_name="Daily_Orders", header=2)
hourly_df = pd.read_excel(file_path, sheet_name="Hourly_Orders", header=2)
product_df = pd.read_excel(file_path, sheet_name="Product_Master", header=2)
metrics_df = pd.read_excel(file_path, sheet_name="Metrics_Template", header=2)

# -------------------------------
# 2) Clean column names
# -------------------------------
daily_df.columns = daily_df.columns.str.strip()
hourly_df.columns = hourly_df.columns.str.strip()
product_df.columns = product_df.columns.str.strip()
metrics_df.columns = metrics_df.columns.str.strip()

# -------------------------------
# 3) Convert timestamp to datetime
# -------------------------------
daily_df["timestamp"] = pd.to_datetime(daily_df["timestamp"])
hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])

# -------------------------------
# 4) Create series_id
#    This helps model understand one product-company series
# -------------------------------
daily_df["series_id"] = daily_df["company_code"] + "_" + daily_df["product_id"]
hourly_df["series_id"] = hourly_df["company_code"] + "_" + hourly_df["product_id"]

# -------------------------------
# 5) Select useful columns for forecasting
# -------------------------------
daily_target = "order_qty"
hourly_target = "order_qty"

daily_covariates = [
    "price",
    "stock_available",
    "promotion",
    "holiday",
    "is_weekend",
    "day_of_week",
    "month",
    "temperature_c",
    "rainfall_mm"
]

hourly_covariates = [
    "price",
    "promotion",
    "holiday",
    "is_weekend",
    "day_of_week",
    "month",
    "hour",
    "business_hour",
    "evening_peak",
    "temperature_c",
    "rainfall_mm"
]

# -------------------------------
# 6) Keep only required columns
# -------------------------------
daily_model_df = daily_df[["timestamp", "series_id", daily_target] + daily_covariates].copy()
hourly_model_df = hourly_df[["timestamp", "series_id", hourly_target] + hourly_covariates].copy()

# -------------------------------
# 7) Sort data
# -------------------------------
daily_model_df = daily_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)
hourly_model_df = hourly_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

# -------------------------------
# 8) Simple train-test split
#    Last 30 days for daily
#    Last 48 hours for hourly
# -------------------------------
daily_train = daily_model_df.groupby("series_id").apply(lambda x: x.iloc[:-30]).reset_index(drop=True)
daily_test  = daily_model_df.groupby("series_id").apply(lambda x: x.iloc[-30:]).reset_index(drop=True)

hourly_train = hourly_model_df.groupby("series_id").apply(lambda x: x.iloc[:-48]).reset_index(drop=True)
hourly_test  = hourly_model_df.groupby("series_id").apply(lambda x: x.iloc[-48:]).reset_index(drop=True)

# -------------------------------
# 9) Check output
# -------------------------------
print("Daily train shape:", daily_train.shape)
print("Daily test shape:", daily_test.shape)
print("Hourly train shape:", hourly_train.shape)
print("Hourly test shape:", hourly_test.shape)

print("\nSample daily data:")
print(daily_train.head())

print("\nSample hourly data:")
print(hourly_train.head())

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1) Load your Excel file
# -------------------------------
file_path = "complex_multi_product_order_forecasting_dataset.xlsx"

daily_df = pd.read_excel(file_path, sheet_name="Daily_Orders", header=2)
hourly_df = pd.read_excel(file_path, sheet_name="Hourly_Orders", header=2)

# -------------------------------
# 2) Clean column names
# -------------------------------
daily_df.columns = daily_df.columns.str.strip()
hourly_df.columns = hourly_df.columns.str.strip()

# -------------------------------
# 3) Convert timestamp column
# -------------------------------
daily_df["timestamp"] = pd.to_datetime(daily_df["timestamp"])
hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])

# -------------------------------
# 4) Create series_id
# -------------------------------
daily_df["series_id"] = daily_df["company_code"].astype(str) + "_" + daily_df["product_id"].astype(str)
hourly_df["series_id"] = hourly_df["company_code"].astype(str) + "_" + hourly_df["product_id"].astype(str)

# -------------------------------
# 5) Choose target and covariates
# -------------------------------
target_col = "order_qty"

daily_covariates = [
    "price",
    "stock_available",
    "promotion",
    "holiday",
    "is_weekend",
    "day_of_week",
    "month",
    "temperature_c",
    "rainfall_mm"
]

hourly_covariates = [
    "price",
    "promotion",
    "holiday",
    "is_weekend",
    "day_of_week",
    "month",
    "hour",
    "business_hour",
    "evening_peak",
    "temperature_c",
    "rainfall_mm"
]

# -------------------------------
# 6) Keep only useful columns
# -------------------------------
daily_model_df = daily_df[["timestamp", "series_id", target_col] + daily_covariates].copy()
hourly_model_df = hourly_df[["timestamp", "series_id", target_col] + hourly_covariates].copy()

# -------------------------------
# 7) Sort data
# -------------------------------
daily_model_df = daily_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)
hourly_model_df = hourly_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

# -------------------------------
# 8) Train-test split
#    daily: last 30 rows per series
#    hourly: last 48 rows per series
# -------------------------------
def split_train_test(df, test_size):
    train_parts = []
    test_parts = []

    for sid, group in df.groupby("series_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        train_parts.append(group.iloc[:-test_size])
        test_parts.append(group.iloc[-test_size:])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df

daily_train, daily_test = split_train_test(daily_model_df, test_size=30)
hourly_train, hourly_test = split_train_test(hourly_model_df, test_size=48)

# -------------------------------
# 9) Separate target and covariates
# -------------------------------
daily_y_train = daily_train[["timestamp", "series_id", target_col]].copy()
daily_x_train = daily_train[["timestamp", "series_id"] + daily_covariates].copy()
daily_x_test = daily_test[["timestamp", "series_id"] + daily_covariates].copy()

hourly_y_train = hourly_train[["timestamp", "series_id", target_col]].copy()
hourly_x_train = hourly_train[["timestamp", "series_id"] + hourly_covariates].copy()
hourly_x_test = hourly_test[["timestamp", "series_id"] + hourly_covariates].copy()

# -------------------------------
# 10) Check output
# -------------------------------
print("Daily train shape:", daily_train.shape)
print("Daily test shape:", daily_test.shape)
print("Hourly train shape:", hourly_train.shape)
print("Hourly test shape:", hourly_test.shape)

print("\nDaily train sample:")
print(daily_train.head())

print("\nHourly train sample:")
print(hourly_train.head())

In [ ]:
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline

# -------------------------------------------------
# 1) Load your Excel file
# -------------------------------------------------
file_path = "complex_multi_product_order_forecasting_dataset.xlsx"

daily_df = pd.read_excel(file_path, sheet_name="Daily_Orders", header=2)
hourly_df = pd.read_excel(file_path, sheet_name="Hourly_Orders", header=2)

# -------------------------------------------------
# 2) Clean column names
# -------------------------------------------------
daily_df.columns = daily_df.columns.str.strip()
hourly_df.columns = hourly_df.columns.str.strip()

# -------------------------------------------------
# 3) Convert timestamp
# -------------------------------------------------
daily_df["timestamp"] = pd.to_datetime(daily_df["timestamp"])
hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])

# -------------------------------------------------
# 4) Create series_id
# -------------------------------------------------
daily_df["series_id"] = daily_df["company_code"].astype(str) + "_" + daily_df["product_id"].astype(str)
hourly_df["series_id"] = hourly_df["company_code"].astype(str) + "_" + hourly_df["product_id"].astype(str)

# -------------------------------------------------
# 5) Select target and covariates
# -------------------------------------------------
target_col = "order_qty"

daily_covariates = [
    "price", "stock_available", "promotion", "holiday",
    "is_weekend", "day_of_week", "month", "temperature_c", "rainfall_mm"
]

hourly_covariates = [
    "price", "promotion", "holiday", "is_weekend",
    "day_of_week", "month", "hour", "business_hour",
    "evening_peak", "temperature_c", "rainfall_mm"
]

# -------------------------------------------------
# 6) Keep only useful columns
# -------------------------------------------------
daily_model_df = daily_df[["timestamp", "series_id", target_col] + daily_covariates].copy()
hourly_model_df = hourly_df[["timestamp", "series_id", target_col] + hourly_covariates].copy()

# -------------------------------------------------
# 7) Sort data
# -------------------------------------------------
daily_model_df = daily_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)
hourly_model_df = hourly_model_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

# -------------------------------------------------
# 8) Simple train-test split
#    daily: last 30 rows per series
#    hourly: last 48 rows per series
# -------------------------------------------------
def split_train_test(df, test_size):
    train_parts = []
    test_parts = []

    for sid, group in df.groupby("series_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        train_parts.append(group.iloc[:-test_size])
        test_parts.append(group.iloc[-test_size:])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df

daily_train, daily_test = split_train_test(daily_model_df, test_size=30)
hourly_train, hourly_test = split_train_test(hourly_model_df, test_size=48)

# -------------------------------------------------
# 9) Prepare Chronos input frames
# -------------------------------------------------
def make_chronos_frames(train_df, test_df, covariate_cols):
    context_df = train_df[["timestamp", "series_id", target_col] + covariate_cols].copy()
    future_df = test_df[["timestamp", "series_id"] + covariate_cols].copy()
    actual_df = test_df[["timestamp", "series_id", target_col]].copy()
    return context_df, future_df, actual_df

daily_context, daily_future, daily_actual = make_chronos_frames(
    daily_train, daily_test, daily_covariates
)

hourly_context, hourly_future, hourly_actual = make_chronos_frames(
    hourly_train, hourly_test, hourly_covariates
)

# -------------------------------------------------
# 10) Load Chronos model
# -------------------------------------------------
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

# -------------------------------------------------
# 11) Forecast function
# -------------------------------------------------
def run_chronos_forecast(context_df, future_df, prediction_length):
    pred_df = pipeline.predict_df(
        context_df=context_df,
        future_df=future_df,
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="series_id",
        timestamp_column="timestamp",
        target=target_col,
    )
    return pred_df

# -------------------------------------------------
# 12) Run forecast for daily and hourly
# -------------------------------------------------
daily_pred = run_chronos_forecast(daily_context, daily_future, prediction_length=30)
hourly_pred = run_chronos_forecast(hourly_context, hourly_future, prediction_length=48)

print("Daily prediction sample:")
print(daily_pred.head())

print("\nHourly prediction sample:")
print(hourly_pred.head())

# -------------------------------------------------
# 13) Merge predictions with actual values
# -------------------------------------------------
def get_point_column(pred_df):
    if "predictions" in pred_df.columns:
        return "predictions"
    elif "0.5" in pred_df.columns:
        return "0.5"
    else:
        return pred_df.columns[-1]

def make_result_table(actual_df, pred_df):
    pred_col = get_point_column(pred_df)
    result = actual_df.merge(
        pred_df[["series_id", "timestamp", pred_col]].copy(),
        on=["series_id", "timestamp"],
        how="left"
    )
    result = result.rename(columns={pred_col: "predicted_order_qty"})
    return result

daily_result = make_result_table(daily_actual, daily_pred)
hourly_result = make_result_table(hourly_actual, hourly_pred)

print("\nDaily result sample:")
print(daily_result.head())

print("\nHourly result sample:")
print(hourly_result.head())

# -------------------------------------------------
# 14) Metrics
# -------------------------------------------------
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def wmape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    if np.sum(np.abs(y_true)) == 0:
        return np.nan
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

# -------------------------------------------------
# 15) Calculate metrics
# -------------------------------------------------
daily_mape = mape(daily_result["order_qty"], daily_result["predicted_order_qty"])
daily_wmape = wmape(daily_result["order_qty"], daily_result["predicted_order_qty"])

hourly_mape = mape(hourly_result["order_qty"], hourly_result["predicted_order_qty"])
hourly_wmape = wmape(hourly_result["order_qty"], hourly_result["predicted_order_qty"])

print("\n--- DAILY METRICS ---")
print("MAPE :", round(daily_mape, 2))
print("WMAPE:", round(daily_wmape, 2))

print("\n--- HOURLY METRICS ---")
print("MAPE :", round(hourly_mape, 2))
print("WMAPE:", round(hourly_wmape, 2))

In [ ]:
import pandas as pd
import numpy as np
import torch
import timesfm

# -------------------------------------------------
# 1) Load your Excel file
# -------------------------------------------------
file_path = "complex_multi_product_order_forecasting_dataset.xlsx"

daily_df = pd.read_excel(file_path, sheet_name="Daily_Orders", header=2)
hourly_df = pd.read_excel(file_path, sheet_name="Hourly_Orders", header=2)

# -------------------------------------------------
# 2) Clean column names
# -------------------------------------------------
daily_df.columns = daily_df.columns.str.strip()
hourly_df.columns = hourly_df.columns.str.strip()

# -------------------------------------------------
# 3) Convert timestamp
# -------------------------------------------------
daily_df["timestamp"] = pd.to_datetime(daily_df["timestamp"])
hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])

# -------------------------------------------------
# 4) Create series_id
# -------------------------------------------------
daily_df["series_id"] = daily_df["company_code"].astype(str) + "_" + daily_df["product_id"].astype(str)
hourly_df["series_id"] = hourly_df["company_code"].astype(str) + "_" + hourly_df["product_id"].astype(str)

# -------------------------------------------------
# 5) Target and covariates
# -------------------------------------------------
target_col = "order_qty"

daily_num_covs = ["price", "stock_available", "temperature_c", "rainfall_mm"]
daily_cat_covs = ["promotion", "holiday", "is_weekend", "day_of_week", "month"]
daily_static_covs = ["company_code", "region", "category", "channel"]

hourly_num_covs = ["price", "temperature_c", "rainfall_mm"]
hourly_cat_covs = ["promotion", "holiday", "is_weekend", "day_of_week", "month", "hour", "business_hour", "evening_peak"]
hourly_static_covs = ["company_code", "region", "category", "channel"]

# -------------------------------------------------
# 6) Train-test split by series
# -------------------------------------------------
def split_by_series(df, horizon):
    train_parts = []
    test_parts = []

    for sid, g in df.groupby("series_id"):
        g = g.sort_values("timestamp").reset_index(drop=True)
        train_parts.append(g.iloc[:-horizon].copy())
        test_parts.append(g.iloc[-horizon:].copy())

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df

daily_train, daily_test = split_by_series(daily_df, horizon=30)
hourly_train, hourly_test = split_by_series(hourly_df, horizon=48)

# -------------------------------------------------
# 7) Prepare TimesFM inputs
#    inputs = past target values
#    dynamic covariates = context + future known covariates
# -------------------------------------------------
def build_timesfm_batches(train_df, test_df, target_col, num_covs, cat_covs, static_covs):
    inputs = []
    dyn_num = {c: [] for c in num_covs}
    dyn_cat = {c: [] for c in cat_covs}
    static_cat = {c: [] for c in static_covs}
    actual_rows = []

    series_ids = sorted(train_df["series_id"].unique())

    for sid in series_ids:
        tr = train_df[train_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        te = test_df[test_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        full = pd.concat([tr, te], ignore_index=True)

        # past target only
        inputs.append(tr[target_col].astype(float).to_numpy())

        # covariates for context + future
        for c in num_covs:
            dyn_num[c].append(full[c].astype(float).to_numpy())
        for c in cat_covs:
            dyn_cat[c].append(full[c].astype(float).to_numpy())

        # static covariates: one value per series
        for c in static_covs:
            static_cat[c].append(str(tr[c].iloc[0]))

        actual_rows.append(te[["timestamp", "series_id", target_col]].copy())

    actual_df = pd.concat(actual_rows, ignore_index=True)
    return inputs, dyn_num, dyn_cat, static_cat, actual_df

daily_inputs, daily_dyn_num, daily_dyn_cat, daily_static_cat, daily_actual = build_timesfm_batches(
    daily_train, daily_test, target_col, daily_num_covs, daily_cat_covs, daily_static_covs
)

hourly_inputs, hourly_dyn_num, hourly_dyn_cat, hourly_static_cat, hourly_actual = build_timesfm_batches(
    hourly_train, hourly_test, target_col, hourly_num_covs, hourly_cat_covs, hourly_static_covs
)

# -------------------------------------------------
# 8) Load TimesFM model
#    TimesFM 2.5 + xreg is needed for covariates
# -------------------------------------------------
torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        fix_quantile_crossing=True,
    )
)

# -------------------------------------------------
# 9) Forecast with covariates
# -------------------------------------------------
def forecast_timesfm(model, inputs, dyn_num, dyn_cat, static_cat, horizon):
    point, quantiles = model.forecast_with_covariates(
        inputs=inputs,
        dynamic_numerical_covariates=dyn_num,
        dynamic_categorical_covariates=dyn_cat,
        static_categorical_covariates=static_cat,
        xreg_mode="xreg + timesfm"
    )
    return point, quantiles

daily_point, daily_q = forecast_timesfm(
    model, daily_inputs, daily_dyn_num, daily_dyn_cat, daily_static_cat, horizon=30
)

hourly_point, hourly_q = forecast_timesfm(
    model, hourly_inputs, hourly_dyn_num, hourly_dyn_cat, hourly_static_cat, horizon=48
)

# -------------------------------------------------
# 10) Build prediction tables
# -------------------------------------------------
def make_pred_table(train_df, test_df, point_forecast):
    rows = []
    series_ids = sorted(train_df["series_id"].unique())

    for i, sid in enumerate(series_ids):
        te = test_df[test_df["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        pred_vals = point_forecast[i]

        tmp = te[["timestamp", "series_id", target_col]].copy()
        tmp["predicted_order_qty"] = pred_vals[:len(tmp)]
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)

daily_result = make_pred_table(daily_train, daily_test, daily_point)
hourly_result = make_pred_table(hourly_train, hourly_test, hourly_point)

print("Daily result sample:")
print(daily_result.head())

print("\nHourly result sample:")
print(hourly_result.head())

# -------------------------------------------------
# 11) Metrics
# -------------------------------------------------
def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def wmape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom * 100

print("\n--- DAILY METRICS ---")
print("MAPE :", round(mape(daily_result[target_col], daily_result["predicted_order_qty"]), 2))
print("WMAPE:", round(wmape(daily_result[target_col], daily_result["predicted_order_qty"]), 2))

print("\n--- HOURLY METRICS ---")
print("MAPE :", round(mape(hourly_result[target_col], hourly_result["predicted_order_qty"]), 2))
print("WMAPE:", round(wmape(hourly_result[target_col], hourly_result["predicted_order_qty"]), 2))

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1) Metric functions
# -----------------------------------
def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def wmape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    total_actual = np.sum(np.abs(y_true))
    if total_actual == 0:
        return np.nan

    return np.sum(np.abs(y_true - y_pred)) / total_actual * 100


def mae(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


# -----------------------------------
# 2) Build metrics table
# -----------------------------------
def build_metrics_table(result_df):
    df = result_df.copy()
    df["abs_error"] = np.abs(df["actual_order_qty"] - df["predicted_order_qty"])
    df["ape"] = np.where(
        df["actual_order_qty"] != 0,
        df["abs_error"] / np.abs(df["actual_order_qty"]),
        np.nan
    )
    return df


# -----------------------------------
# 3) Evaluate one forecast result
# -----------------------------------
def evaluate_forecast(result_df, name="Forecast"):
    df = build_metrics_table(result_df)

    y_true = df["actual_order_qty"]
    y_pred = df["predicted_order_qty"]

    score = {
        "name": name,
        "MAE": mae(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "WMAPE": wmape(y_true, y_pred),
    }

    print(f"\n--- {name} ---")
    print("MAE  :", round(score["MAE"], 2))
    print("MAPE :", round(score["MAPE"], 2))
    print("WMAPE:", round(score["WMAPE"], 2))

    return df, score


# -----------------------------------
# 4) Example usage
#    daily_result and hourly_result
#    must already exist
# -----------------------------------
daily_metrics_df, daily_scores = evaluate_forecast(daily_result, "Daily Forecast")
hourly_metrics_df, hourly_scores = evaluate_forecast(hourly_result, "Hourly Forecast")

# -----------------------------------
# 5) Save output files
# -----------------------------------
daily_metrics_df.to_csv("daily_forecast_metrics.csv", index=False)
hourly_metrics_df.to_csv("hourly_forecast_metrics.csv", index=False)

print("\nSaved:")
print("- daily_forecast_metrics.csv")
print("- hourly_forecast_metrics.csv")